# Test: tvärsnittsklassning (tvk)

Kontrollerar `kb.tvk` mot EN 1993-1-1 tabell 5.2. Funktionerna returnerar klassen (1–4). `klass()` kontrollerar också att den utskrivna klassen är samma som den returnerade.

Kör med **Run All**. Kerneln ska vara `.venv` i repot.

- ✔ kontrollen gick igenom
- ✘ känt fel: fel som redan är kända och inte rättade än
- ✘ FEL: nytt fel

In [ ]:
import math
import kthbygg as kb
from kontroll import kolla, kolla_fel, klass, utskrift, sammanfattning, nollställ

nollställ()

## Inre tryckt del

Gränser 33ε, 38ε, 42ε.

In [ ]:
kolla("tryck_inre c/t=20, S235", lambda: klass(kb.tvk.tryck_inre, 200, 10, 235), 1)
kolla("tryck_inre c/t=33,3, S275", lambda: klass(kb.tvk.tryck_inre, 100, 3, 275), 2)  # 30,5 < 33,3 ≤ 35,1
kolla("tryck_inre c/t=40, S235", lambda: klass(kb.tvk.tryck_inre, 400, 10, 235), 3)
kolla("tryck_inre c/t=50, S235", lambda: klass(kb.tvk.tryck_inre, 500, 10, 235), 4)

## Inre böjd del

Gränser 72ε, 83ε, 124ε. För S355 är ε = 0,814.

In [ ]:
kolla("böj_inre c/t=50, S355", lambda: klass(kb.tvk.böj_inre, 500, 10, 355), 1)
kolla("böj_inre c/t=75, S355", lambda: klass(kb.tvk.böj_inre, 600, 8, 355), 3)   # 67,5 < 75 ≤ 100,9
kolla("böj_inre c/t=110, S355", lambda: klass(kb.tvk.böj_inre, 880, 8, 355), 4)

## Inre del, tryck och böjning

α = 0,5 ger gränserna 36ε/α och 41,5ε/α. Med S355 och c/t = 62 blir det klass 2 (58,6 < 62 ≤ 67,5).

Klass 3 beror på ψ. Med α = 0,6 och c/t = 100 i S235: ψ = −1 ger gränsen 62ε(1−ψ)√(−ψ) = 124, alltså klass 3. ψ = 0 ger 42ε/0,67 = 62,7, alltså klass 4. Därför måste ψ alltid anges.

In [ ]:
kolla("tryck_böj_inre α=0,5, c/t=40, S275", lambda: klass(kb.tvk.tryck_böj_inre, 400, 10, 275, 200, 0), 1)
kolla("tryck_böj_inre α=0,5, c/t=62, S355", lambda: klass(kb.tvk.tryck_böj_inre, 620, 10, 355, 310, 0), 2)
kolla("tryck_böj_inre α=0,6, c/t=100, ψ=−1", lambda: klass(kb.tvk.tryck_böj_inre, 1000, 10, 235, 600, -1), 3)
kolla("tryck_böj_inre α=0,6, c/t=100, ψ=0", lambda: klass(kb.tvk.tryck_böj_inre, 1000, 10, 235, 600, 0), 4)
kolla_fel("tryck_böj_inre utan ψ", lambda: kb.tvk.tryck_böj_inre(400, 10, 275, 200), TypeError)
kolla_fel("tryck_böj_inre αc > c", lambda: kb.tvk.tryck_böj_inre(400, 10, 275, 500, 0))

## Utstående del, tryck

Gränser 9ε, 10ε, 14ε.

In [ ]:
kolla("tryck_frikant c/t=6,7, S275", lambda: klass(kb.tvk.tryck_frikant, 20, 3, 275), 1)
kolla("tryck_frikant c/t=10, S235", lambda: klass(kb.tvk.tryck_frikant, 40, 4, 235), 2)
kolla("tryck_frikant c/t=15, S235", lambda: klass(kb.tvk.tryck_frikant, 60, 4, 235), 4)

## Utstående del, tryck och böjning

α = αc/c är den tryckta andelen (≤ 1). Med c = 80, αc = 40 är α = 0,5:

- spetsen tryckt: klass 1 om c/t ≤ 9ε/α = 18
- spetsen dragen: klass 1 om c/t ≤ 9ε/(α√α) = 25,5

α utanför 0 < α ≤ 1 ska ge ett tydligt fel.

In [ ]:
kolla("tip_tryck_frikant α=0,5, c/t=10", lambda: klass(kb.tvk.tip_tryck_frikant, 80, 8, 235, 40, 0.57), 1)
kolla("tip_drag_frikant α=0,5, c/t=10", lambda: klass(kb.tvk.tip_drag_frikant, 80, 8, 235, 40, 0.57), 1)
kolla("tip_tryck_frikant α=1 som rent tryck", lambda: klass(kb.tvk.tip_tryck_frikant, 80, 8, 235, 80, 0.43), 2)
kolla_fel("tip_tryck_frikant αc > c", lambda: kb.tvk.tip_tryck_frikant(80, 8, 235, 100, 0.57))
kolla_fel("tip_drag_frikant αc = 0", lambda: kb.tvk.tip_drag_frikant(80, 8, 235, 0, 0.57))

## Vinkelstång

Klass 3 kräver både h/t ≤ 15ε (längsta benet) och (b+h)/2t ≤ 11,5ε, annars klass 4.

In [ ]:
kolla("vinkelstång 80×80×8: båda uppfyllda", lambda: klass(kb.tvk.vinkelstång, 80, 80, 8, 235), 3)
kolla("vinkelstång 100×100×8: (b+h)/2t = 12,5", lambda: klass(kb.tvk.vinkelstång, 100, 100, 8, 235), 4)
kolla("vinkelstång 160×40×10: h/t = 16", lambda: klass(kb.tvk.vinkelstång, 160, 40, 10, 235), 4)
kolla("vinkelstång 40×160×10: ordningen spelar ingen roll", lambda: klass(kb.tvk.vinkelstång, 40, 160, 10, 235), 4)
kolla("vinkelstång skriver inte 'plastisk' för klass 3",
      lambda: "plastisk" in utskrift(kb.tvk.vinkelstång, 80, 80, 8, 235), False)

## Cirkulärt rör

d/t ≤ 50ε², 70ε², 90ε².

In [ ]:
kolla("cirkulärt_rör d/t=33, S235", lambda: klass(kb.tvk.cirkulärt_rör, 100, 3, 235), 1)
kolla("cirkulärt_rör d/t=100, S275", lambda: klass(kb.tvk.cirkulärt_rör, 100, 1, 275), 4)

## Sammanfattning

Cellen stoppar med `AssertionError` om någon kontroll som inte är märkt som känt fel misslyckas.

In [ ]:
sammanfattning()